# Interactive Agent Session: Chapter 4 — CrewAI Autonomous Marketing Swarm

**The Business Problem:** Launching a 360° marketing campaign requires research, strategy, copywriting, and brand review — typically 4 different hires working over 2 weeks.

**The Solution:** CrewAI orchestrates 4 specialized agents in a sequential pipeline. Each agent builds on the previous output. Total execution: ~30 seconds.

**Key Concept:** CrewAI automates the **handoff pattern** — Agent A finishes → output passes to Agent B → etc.

In [ ]:
import sys
!{sys.executable} -m pip install --quiet crewai python-dotenv

In [ ]:
%%capture crew_logs

import os
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process

load_dotenv()
MODEL_ID = "gpt-4o-mini"
USE_SIMULATION = not os.getenv("OPENAI_API_KEY")

# ── THE CREW ──
researcher = Agent(role="Market Intelligence Analyst", goal="Identify the top 3 consumer pain points for smart home security devices.", backstory="Former investigative journalist at Wired, now specializing in consumer tech insights.", llm=MODEL_ID, verbose=False)
strategist = Agent(role="Chief Marketing Strategist", goal="Craft a killer campaign concept based on research insights.", backstory="Ex-Nike brand strategist with 15 years in emotional marketing. Known for viral campaigns.", llm=MODEL_ID, verbose=False)
copywriter = Agent(role="Senior Copywriter", goal="Write the final ad copy: headline, tagline, and 3-sentence body.", backstory="Award-winning copywriter. Cannes Lions winner. Master of short-form persuasion.", llm=MODEL_ID, verbose=False)
brand_reviewer = Agent(role="Brand Compliance Officer", goal="Review the final copy for legal and brand guideline compliance.", backstory="Former legal counsel at P&G. Ensures all marketing meets FTC and brand standards.", llm=MODEL_ID, verbose=False)

# ── THE TASKS ──
task1 = Task(description="Research the top 3 consumer pain points for smart home security systems. Focus on real frustrations from Amazon reviews and Reddit.", agent=researcher, expected_output="3 numbered pain points with evidence.")
task2 = Task(description="Based on the research, create a campaign concept. Include: campaign name, emotional hook, and target audience.", agent=strategist, expected_output="Campaign name, hook, and target persona.")
task3 = Task(description="Write the final ad copy. Include: 1 headline (max 8 words), 1 tagline, and a 3-sentence body paragraph.", agent=copywriter, expected_output="Headline, tagline, and body copy.")
task4 = Task(description="Review the ad copy for FTC compliance, trademark issues, and brand tone consistency. Flag any problems.", agent=brand_reviewer, expected_output="Compliance review with APPROVED or REJECTED status.")

crew = Crew(agents=[researcher, strategist, copywriter, brand_reviewer], tasks=[task1, task2, task3, task4], process=Process.sequential, verbose=False)

if USE_SIMULATION:
    research_out = "1. False alarms at 3AM erode trust — 67% of users report at least one per month\n2. Monthly subscription fees ($10-30) feel predatory after buying expensive hardware\n3. Complex multi-app setup deters 40% of non-technical buyers"
    strategy_out = "Campaign: SILENT GUARDIAN\nHook: Your home security should protect your sleep, not interrupt it.\nTarget: Millennial homeowners (28-40) who value simplicity over features."
    copy_out = "Headline: Protection That Never Wakes You Up\nTagline: Zero false alarms. Zero monthly fees. Zero stress.\nBody: Your home deserves a guardian that works silently in the background. SmartPass 2.0 uses AI-powered detection to eliminate false alarms entirely. Setup takes 60 seconds. No subscriptions, no complexity, no compromise."
    review_out = "STATUS: APPROVED ✅\nFTC Check: No unsubstantiated claims. AI-powered is demonstrable.\nTrademark: SmartPass 2.0 — clear, no conflicts found.\nTone: On-brand. Calm, confident, consumer-friendly."
else:
    result = crew.kickoff()
    research_out = str(task1.output)
    strategy_out = str(task2.output)
    copy_out = str(task3.output)
    review_out = str(task4.output)

In [ ]:
from IPython.display import display, HTML

def card(role, icon, content):
    return '<div style="background:#f7fafc; padding:25px; border-radius:24px; border:1px solid #edf2f7;"><div style="font-size:10px; font-weight:800; text-transform:uppercase; letter-spacing:2px; color:#ed64a6; margin-bottom:12px;">' + icon + ' ' + role + '</div><div style="font-size:13px; color:#2d3748; line-height:1.7;">' + content.replace("\n", "<br>") + '</div></div>'

html = '<style>@import url("https://fonts.googleapis.com/css2?family=Unbounded:wght@800&family=Outfit:wght@300;600&display=swap");</style>'
html += '<div style="max-width:950px; margin:30px auto; font-family:Outfit,sans-serif; background:#fff; padding:60px; border-radius:50px; border:1px solid #eee; box-shadow:0 40px 120px rgba(0,0,0,0.08); color:#1a202c;">'
html += '<div style="font-family:Unbounded,sans-serif; font-size:20px; color:#1a202c; letter-spacing:-1px; margin-bottom:40px;">CREW_STRATEGY_DECK: SMART_SEC_2.0</div>'
html += '<div style="display:grid; grid-template-columns:1fr 1fr; gap:25px;">'
html += card("Research Intel", "🔍", research_out)
html += card("Strategy Hook", "🎯", strategy_out)
html += card("Ad Copy", "✍️", copy_out)
html += card("Compliance Review", "🛡️", review_out)
html += '</div></div>'
display(HTML(html))